t-Testing

In [14]:
import pandas as pd
import numpy as np
from scipy import stats

Load the Data

In [15]:
df = pd.read_csv('../../output/04_Regression_handled.csv')

Daily Abnormal Returns

In [16]:
df['AR'] = df['R_i'] - (df['alpha'] + df['beta'] * df['R_m'])

Event Window

In [17]:
windows = {
    'CAR(-5,-1)': (-5, -1),
    'CAR(0,0)': (0, 0),
    'CAR(0,+5)': (0, 5),
    'CAR(-5,+30)': (-5, 30)
}

t-test Function

In [18]:
def run_cross_sectional_ttest(data, start_day, end_day, window_name):
    #Filter the data
    window_data = data[(data['day'] >= start_day) & (data['day'] <= end_day)]

    #sum daily ARto get CAR
    stock_cars = window_data.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
    stock_cars.rename(columns={'AR': 'CAR'}, inplace=True)

    results = []

    # Sector level t-test
    sectors = stock_cars['sector'].unique()

    for sector in sectors:
        sector_data = stock_cars[stock_cars['sector'] == sector]['CAR']
        n_stocks = len(sector_data)
        
        if n_stocks > 1:
            mean_car = sector_data.mean()
            std_car = sector_data.std()
            t_stat, p_val = stats.ttest_1samp(sector_data, popmean=0)
        else:
            mean_car = sector_data.mean()
            std_car = np.nan
            t_stat = np.nan
            p_val = np.nan

        results.append({
            'Window': window_name,
            'Level': 'Sector',
            'Name': sector,
            'N': n_stocks,
            'Mean_CAR(%)': mean_car * 100, # Convert to percentage for readability
            'Std_Dev(%)': std_car * 100,
            't-statistic': t_stat,
            'p-value': p_val,
            'Significant_at_5%': p_val < 0.05 if pd.notna(p_val) else False
        })

    #Market t-test
    all_cars = stock_cars['CAR']
    n_total = len(all_cars)
    t_stat_mkt, p_val_mkt = stats.ttest_1samp(all_cars, popmean=0)

    results.append({
        'Window': window_name,
        'Level': 'Market',
        'Name': 'ALL STOCKS',
        'N': n_total,
        'Mean_CAR(%)': all_cars.mean() * 100, 
        'Std_Dev(%)': all_cars.std() * 100,
        't-statistic': t_stat_mkt,
        'p-value': p_val_mkt,
        'Significant_at_5%': p_val_mkt < 0.05 
    })

    return pd.DataFrame(results)



Test all windows

In [19]:
all_results = []

for window_name, (start, end) in windows.items():
    window_results_df = run_cross_sectional_ttest(df, start, end, window_name)
    all_results.append(window_results_df)

final_ttest_results = pd.concat(all_results,ignore_index=True)

formatted_results = final_ttest_results.copy()
formatted_results['Mean_CAR(%)'] = formatted_results['Mean_CAR(%)'].round(2)
formatted_results['Std_Dev(%)'] = formatted_results['Std_Dev(%)'].round(2)
formatted_results['t-statistic'] = formatted_results['t-statistic'].round(3)
formatted_results['p-value'] = formatted_results['p-value'].round(4)

Results

In [20]:
display_df = formatted_results[formatted_results['Window'] == 'CAR(-5,+30)'].sort_values(by='Mean_CAR(%)')
print(display_df[['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']])


formatted_results.to_csv('../../output/05_ttest_results.csv', index=False)

                          Name    N  Mean_CAR(%)  t-statistic  p-value  \
81                      Energy    2       -52.38       -1.041   0.4871   
64                   Insurance   12       -38.50       -1.196   0.2570   
73                   Retailing   14       -18.33       -1.122   0.2821   
68                   Materials   21       -12.98       -0.715   0.4831   
77              Transportation    3       -10.03       -1.160   0.3660   
72                 Real Estate   17        -9.42       -2.552   0.0213   
79         Software & Services    1        -7.41          NaN      NaN   
78  Telecommunication Services    2        -5.34       -1.711   0.3368   
75         Commercial Services    7        -2.98       -0.441   0.6746   
65           Consumer Durables    8        -2.28       -0.620   0.5549   
74          Household Products    2        -0.61       -1.000   0.5000   
70           Consumer Services   32        -0.56       -0.143   0.8871   
71        Healthcare Equipment    8   